In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": { "parameterName": "LakehouseName", "defaultValue": "AzureCostAnalyzer_LH" },
    "id": { "parameterName": "LakehouseId", "defaultValue": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx" },
    "workspaceId": { "parameterName": "WorkspaceId", "defaultValue": "yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy" }
  }
}

# 04 · Gold · Cost Summary

Builds **`gold_cost_summary_daily`** and **`gold_cost_summary_monthly`** from `focus_partitioned`.

## Output tables
| Table | Grain | Dimensions | Measures |
|---|---|---|---|
| `gold_cost_summary_daily` | Date × Sub × Service | SubAccountName, ServiceCategory, ServiceName, RegionName | EffectiveCost, BilledCost, ListCost, SavingsAmount |
| `gold_cost_summary_monthly` | YearMonth × Sub × Service | SubAccountName, ServiceCategory, ServiceName | EffectiveCost, BilledCost, ListCost, SavingsAmount |

**Filter**: `ChargeCategory = 'Usage'` only (excludes Purchase/Tax/Adjustment/Credit).

**Dependencies**: `focus_partitioned` (silver).

In [ ]:
from pyspark.sql import functions as F

In [ ]:
# Parameters
SOURCE_TABLE = "focus_partitioned"
DAILY_TABLE   = "gold_cost_summary_daily"
MONTHLY_TABLE = "gold_cost_summary_monthly"

## Step 1 · Build `gold_cost_summary_daily`

Daily grain: Date × SubAccountName × ServiceCategory × ServiceName × RegionName.

In [ ]:
# Partition pruning: read last 12 months only
from datetime import date
from dateutil.relativedelta import relativedelta

lookback_date = date.today() - relativedelta(months=12)

daily_df = (
    spark.table(SOURCE_TABLE)
         .filter(
             (F.col("Year") > lookback_date.year) | 
             ((F.col("Year") == lookback_date.year) & (F.col("Month") >= lookback_date.month))
         )
         .filter(F.col("ChargeCategory") == "Usage")  # Exclude Purchase/Tax/etc
         .withColumn("Date", F.to_date("ChargePeriodStart"))
         .groupBy("Date", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName")
         .agg(
             F.sum("EffectiveCost").alias("EffectiveCost"),
             F.sum("BilledCost").alias("BilledCost"),
             F.sum("ListCost").alias("ListCost"),
             F.sum(F.col("ListCost") - F.col("EffectiveCost")).alias("SavingsAmount"),
         )
)

daily_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DAILY_TABLE)
print(f"✓ Created {DAILY_TABLE} with {daily_df.count():,} rows")

## Step 2 · Build `gold_cost_summary_monthly`

Monthly grain: YearMonth × SubAccountName × ServiceCategory × ServiceName.

In [ ]:
monthly_df = (
    spark.table(SOURCE_TABLE)
         .filter(
             (F.col("Year") > lookback_date.year) | 
             ((F.col("Year") == lookback_date.year) & (F.col("Month") >= lookback_date.month))
         )
         .filter(F.col("ChargeCategory") == "Usage")
         .withColumn("YearMonth", F.date_format("ChargePeriodStart", "yyyy-MM"))
         .groupBy("YearMonth", "SubAccountName", "ServiceCategory", "ServiceName")
         .agg(
             F.sum("EffectiveCost").alias("EffectiveCost"),
             F.sum("BilledCost").alias("BilledCost"),
             F.sum("ListCost").alias("ListCost"),
             F.sum(F.col("ListCost") - F.col("EffectiveCost")).alias("SavingsAmount"),
         )
)

print(f"  Total Effective Cost: ${monthly_df.agg(F.sum('EffectiveCost')).first()[0]:,.2f}")

monthly_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(MONTHLY_TABLE)
print(f"✓ Created {MONTHLY_TABLE} with {monthly_df.count():,} rows")